# Auditing components.yaml

This notebook runs the AuditSystem against the main components.yaml file to evaluate solution quality.

In [1]:
from pathlib import Path

import ibis
from hamilton import driver, base

from iacs.audit_system import AuditRunner
from iacs.transforms import manifest_to_registry
from iacs.transform_system import TransformSystem

ibis.options.interactive = True  # auto-display results (like pandas)

## Load components.yaml

In [2]:
COMPONENTS_DIR = str(Path("../components"))

# Build registry from YAML manifests using Hamilton DAG
build_driver = driver.Driver(
    {"input_dir": COMPONENTS_DIR},
    manifest_to_registry,
    adapter=base.DictResult(),
)
result = build_driver.execute(["registry"])
reg = result["registry"]

# Create TransformSystem for post-registry transforms
ts = TransformSystem(reg, [manifest_to_registry])

print(f"Component types: {reg.component_types}")
print(f"Number of component types: {len(reg.component_types)}")
print(f"Available transforms: {ts.outputs}")

Component types: ['name', 'description', 'file_info', 'requirement', 'parent', 'id', 'todo', 'system', 'input', 'output', 'weight']
Number of component types: 11
Available transforms: ['complete_schema', 'component_first_data', 'schema', 'parent', 'components_database', 'conn', 'components', 'data_models', 'flattened_entity_first_data', 'flattened_data', 'name_to_id', 'raw_entity_first_data', 'validated_components']


## Run All Audits

In [3]:
runner = AuditRunner.default()

results = runner.run(reg)

print(f"All audits passed: {runner.all_passed}")
print()
for name, table in results.items():
    count = table.count().execute()
    status = "PASS" if count == 0 else "FAIL"
    print(f"{name}: {status} ({count} issues)")

All audits passed: False

requirement_coverage: FAIL (43 issues)
traceability: FAIL (81 issues)
todo: FAIL (16 issues)


## Requirement Coverage Audit

Checks that all requirements have implementing solutions.

In [8]:
reg.view(["requirement", "description", "id"])

┏━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ entity_id    ┃ priority ┃ requirement.value ┃ description.value                                                                ┃ value                          ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ string       │ float64  │ string            │ string                                                                           │ string                         │
├──────────────┼──────────┼───────────────────┼──────────────────────────────────────────────────────────────────────────────────┼────────────────────────────────┤
│ 496fe4089c80 │     NULL │ NULL              │ One potential solution for allowing users to define infrastructure is via an EC… │ ecs_framework_base_requirement │
│ 44371d6deb86 │     NULL │ NULL              │ The requirements specifically for our ECS framework.                             │ ecs_framework_requirements     │
│ 629588dd0b7c │     NULL │ NULL              │ Our specific approach to requirements is to start with the fundamental requirem… │ 629588dd0b7c                   │
│ fffe9a95f46a │     NULL │ NULL              │ Any requirement depending on another requirement has a parent relationship to t… │ fffe9a95f46a                   │
│ 4a951849dfc5 │     NULL │ NULL              │ A generalized requirement of several other requirements is the ability to descr… │ ecs_hierarchical_requirement   │
└──────────────┴──────────┴───────────────────┴──────────────────────────────────────────────────────────────────────────────────┴────────────────────────────────┘

In [4]:
results["requirement_coverage"]

┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┓
┃ entity_id    ┃ description                                                                      ┃ priority ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━┩
│ string       │ string                                                                           │ float64  │
├──────────────┼──────────────────────────────────────────────────────────────────────────────────┼──────────┤
│ 73287cb6715e │ We don't want to duplicate other IaC code if possible.\n                         │     NULL │
│ cb56c60ed680 │ Code such as Python also provides clear definitions.\n                           │     NULL │
│ 8286be4b5f2d │ Version control is essential for reliability.\n                                  │     NULL │
│ 2ee039d50895 │ A source of truth is not *a* source of truth if it is distributed.\n             │     NULL │
│ 496fe4089c80 │ One potential solution for allowing users to define infrastructure is via an EC… │     NULL │
│ 39703077f4ef │ We need to be able to define standard parts of solutions architecture, e.g. use… │     NULL │
│ 19e19cf8eeba │ Human-executed infrastructure is infrastructure that is not yet (or cannot be) … │     NULL │
│ 559968d335a5 │ Abstract components of infrastructure include poorly-defined components and abs… │     NULL │
│ a20f524005c3 │ We of course want the data format to be compatible with expensive calculations,… │      0.1 │
│ c86532577f5f │ Wikis are a very common and readable way of documenting knowledge. This would b… │     NULL │
│ …            │ …                                                                                │        … │
└──────────────┴──────────────────────────────────────────────────────────────────────────────────┴──────────┘

## Traceability Audit

Checks that all entities trace back to requirements.

In [5]:
results["traceability"]

┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ entity_id    ┃ message                                                  ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ string       │ string                                                   │
├──────────────┼──────────────────────────────────────────────────────────┤
│ 8a971e033ead │ Entity '8a971e033ead' does not trace to any requirement. │
│ beab9d15329e │ Entity 'beab9d15329e' does not trace to any requirement. │
│ 6985ca1f4daa │ Entity '6985ca1f4daa' does not trace to any requirement. │
│ 43ec9c8459ec │ Entity '43ec9c8459ec' does not trace to any requirement. │
│ 9f4bda12c1ff │ Entity '9f4bda12c1ff' does not trace to any requirement. │
│ 6c00a71d6bfc │ Entity '6c00a71d6bfc' does not trace to any requirement. │
│ 8fca07d1f563 │ Entity '8fca07d1f563' does not trace to any requirement. │
│ 790b34b5a3aa │ Entity '790b34b5a3aa' does not trace to any requirement. │
│ 7df306c4eda2 │ Entity '7df306c4eda2' does not trace to any requirement. │
│ 380f1c590d32 │ Entity '380f1c590d32' does not trace to any requirement. │
│ …            │ …                                                        │
└──────────────┴──────────────────────────────────────────────────────────┘

## Todo Audit

Reports outstanding todos.

In [6]:
results["todo"]

┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ entity_id    ┃ value                                                                            ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ string       │ string                                                                           │
├──────────────┼──────────────────────────────────────────────────────────────────────────────────┤
│ f8f0ca6cbfab │ Have files be the topmost entities.\n                                            │
│ f8f0ca6cbfab │ Notation for multiple components of the same type. Maybe "component_type s"?\n   │
│ f8f0ca6cbfab │ Binary relationship notation going both ways. Maybe "component_type of:" and "c… │
│ f8f0ca6cbfab │ Notation for indicating that a component is a row-wise subset of another compon… │
│ f8f0ca6cbfab │ Notation for defining an entity in-line vs linking to an existing entity. Maybe… │
│ f8f0ca6cbfab │ Notation for indicating that a component must be joined with another component … │
│ f8f0ca6cbfab │ Notation for working with a component that would be a subset of another compone… │
│ f8f0ca6cbfab │ Notation for subsets with constant values for one or more fields.\n              │
│ f8f0ca6cbfab │ Notation for functions and classes.\n                                            │
│ f8f0ca6cbfab │ Notation for a component that is at the intersection of two subsets.\n           │
│ …            │ …                                                                                │
└──────────────┴──────────────────────────────────────────────────────────────────────────────────┘